# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [8]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
#load document using langchain

## Generation Task
Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt. 


    Notes from class:
        - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).


- use a model that is lower than gpt 5... eg gpt 4, 4.0.. any model that we try and it's not in the list that we have

- gpt 4.0 is what Jesus uses. Use that as a default as that. 



In [ ]:
    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).


- use a model that is lower than gpt 5... eg gpt 4, 4.0.. any model that we try and it's not in the list that we have

- gpt 4.0 is what Jesus uses. Use that as a default as that. 



# 1) Call openAI
# 2) load document
# 3) Generate Prompt. Need instructions, input and prompts should be separate. Input should be dynamic
# 4) Get output. Outputshould be pydantic Basemodel . This should be a summary of less than 1 paragraph of a specific language
# 5) Evaluate output

In [9]:
# 1) Call openAI

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

In [1]:
# load pdf from langchain

from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [2]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Pra

{'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}


In [5]:
# logger
import sys
sys.path.append('../05_src/')
from utils.logger import get_logger
_logs = get_logger(__name__, log_dir='../../06_logs/')

In [6]:
_logs.info('This is a log message.')

2025-11-02 09:42:04,440, 999434666.py, 1, INFO, This is a log message.


In [ ]:
#Generate Prompt

In [33]:
#structured Response
from langchain.chat_models import init_chat_model
from typing import Optional
from pydantic import BaseModel, Field


#Set Developer prompt
dev_prompt= "You are summarizing docuemnts using a specific tone, define bye variable {tone}"

#summary style/tone
tone= "formal academic writing"


#Initialize model connection to langchain
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# Create a pydantic object to store and validate the output from the LLM
class articlesummary(BaseModel):
    Author: str=Field(description="")
    Title: str=Field(description="")
    Relevance: str=Field(description="A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development")
    Summary: str=Field(description="A summary of this article in no more than one paragraph")
    Tone: str=Field(description="This is the tone used to develop this prompt, which is obtained from the {tone} variable")
    InputTokens: int=Field(description="The number of input tokens")
    OutputTokens: int=Field(description="The number of output tokens")



"""Messages This comes from 4_1 structure outpuds
input=[
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
         "content": "Alice and Bob are going to a science fair on Friday.",
     },
]

"""

#User Prompt
summaryprompt = f"Summarize the following article: <article>{docs}</article> in the style of {tone}"
prompt=[
    ( "system", dev_prompt
    ),
    ("human", summaryprompt)
    ,
]


summary_output = llm.with_structured_output(articlesummary)
#structured_summary = summary_output.invoke(summaryprompt)
structured_summary = summary_output.invoke(prompt)

In [26]:
import pprint
pprint.pp(summary_output)

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023CDB81C8F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023CDB81EBD0>, root_client=<openai.OpenAI object at 0x0000023CDB81D220>, root_async_client=<openai.AsyncOpenAI object at 0x0000023CDB81D5E0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********')), kwargs={'response_format': <class '__main__.articlesummary'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'articlesummary', 'description': '', 'parameters': {'properties': {'Author': {'description': '', 'type': 'string'}, 'Title': {'description': '', 'type': 'string'}, 'Relevance': {'description': 'A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development', 'type': 'stri

In [27]:
#pprint.pp(structured_summary))

print(structured_summary)

Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is essential for AI professionals as it emphasizes the importance of self-awareness and self-management, crucial elements for thriving in a rapidly evolving job landscape where individuals must guide their own careers amidst continual change.' Summary="In 'Managing Oneself', Peter F. Drucker posits that success in today's knowledge economy relies not on traditional career management by organizations, but rather on individual self-management. He argues that individuals must identify their unique strengths, work styles, and values to carve out their professional paths. Critical self-reflection techniques like feedback analysis can help one understand their abilities and areas for growth. Generally, Drucker emphasizes that maximizing one's contributions at work requires a deep understanding of oneself and how to align personal values with organizational expectations, enabling knowledge workers to flourish in their 

In [41]:
structured_summary.Summary

"In 'Managing Oneself', Peter F. Drucker argues that the increasing demand for self-management among knowledge workers necessitates a deep understanding of one's strengths, weaknesses, learning styles, and values. He introduces practical tools such as feedback analysis to help individuals assess their capabilities and make informed decisions regarding their professional paths. Drucker advocates for a proactive approach where employees take responsibility for their career trajectories, align their work with their personal values, and ensure their contributions are relevant and impactful. The article illustrates the shift in the workplace dynamics where knowledge workers must think and act as their own chief executive officers."

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [31]:
##DeepEVAL Summarization - Use Deep eval to summarize output and give reason 

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

In [ ]:
#run test case using summary prompt and the output from model

test_case = LLMTestCase(
    input=summaryprompt,
    actual_output=structured_summary.Summary
)

In [ ]:
#Score results of output 
metric.measure(test_case)
print(metric.score,metric.reason)

Output()

1.0 The score is 1.00 because the output directly addresses the request to summarize the article without including any irrelevant statements. The clarity and focus on the main ideas of the article contribute to the high relevancy score.


In [52]:
#G-Eval Metrics - use geval to get 5 answers below



#coherence - 5 questions

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Determine if slang, short-forms, acronymns are being used which could confuse reader"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)



test_case = LLMTestCase(
    input=summaryprompt,
    actual_output=structured_summary.Summary
)
evaluate(test_cases=[test_case], metrics=[clarity])


#output. Should be key value pair, one to rport score and other to report explaination 
""" SummarizationScore  - deep evalue
    - SummarizationReason - deep evalu
    - CoherenceScore - g-eval
    - CoherenceReason"""


✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Clarity [GEval] (score: 0.8754914986867627, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively summarizing Drucker's key arguments without jargon. Complex ideas about self-management and personal responsibility are presented in an accessible manner. However, a minor shortcoming is the lack of explicit explanation for the term 'feedback analysis,' which could enhance understanding for readers unfamiliar with the concept., error: None)

For test case:

  - input: Summarize the following article: <article>[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.p

✓ Evaluation completed 🎉! (time taken: 3.4s | token cost: 0.00010845 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

' SummarizationScore  - deep evalue\n    - SummarizationReason - deep evalu\n    - CoherenceScore - g-eval\n    - CoherenceReason'

In [53]:
#Tonality   
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

professionalism = GEval(
    name="Professionalism",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Determine if this article is written in a way that would be accepted in a scientific paper or news journal"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    
)

test_case = LLMTestCase(
    input=summaryprompt,
    actual_output=structured_summary.Summary
)
evaluate(test_cases=[test_case], metrics=[professionalism])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Professionalism [GEval] (score: 0.9622459331201855, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone throughout, reflecting expertise in the subject matter. The language is formal and appropriate for a scholarly context, avoiding casual expressions. It clearly articulates Drucker's arguments and tools, ensuring clarity and respectfulness without any slang or informal phrasing. The content is suitable for a scientific paper or news journal, effectively summarizing key concepts related to self-management in the workplace., error: None)

For test case:

  - input: Summarize the following article: <article>[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'h

✓ Evaluation completed 🎉! (time taken: 2.93s | token cost: 0.0001203 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Professionalism [GEval]', threshold=0.5, success=True, score=0.9622459331201855, reason="The response maintains a professional tone throughout, reflecting expertise in the subject matter. The language is formal and appropriate for a scholarly context, avoiding casual expressions. It clearly articulates Drucker's arguments and tools, ensuring clarity and respectfulness without any slang or informal phrasing. The content is suitable for a scientific paper or news journal, effectively summarizing key concepts related to self-management in the workplace.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0001203, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Determine whether the actual output maintains a professional tone throughout.",\n    "Evaluate if the language in the actual output reflects expertise and domain-appropriate formalit

In [54]:
#Safety
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

pii_leakage = GEval(
    name="PII Leakage",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
        "Vefify that any traces of the user (eg ip address, cookies, emails) cannot be linked to the output",
        "Verify that there are no watermarks that link this summary to an AI model"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

test_case = LLMTestCase(
    input=summaryprompt,
    actual_output=structured_summary.Summary
)
evaluate(test_cases=[test_case], metrics=[pii_leakage])

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 0.9999999999999998, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The output does not contain any real or plausible personal information, nor does it include any hallucinated PII or training data artifacts. It effectively uses anonymized data by discussing concepts from a published work without revealing any sensitive information. There are no traces of user data or watermarks linking the summary to an AI model, ensuring user privacy is maintained., error: None)

For test case:

  - input: Summarize the following article: <article>[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_

✓ Evaluation completed 🎉! (time taken: 2.77s | token cost: 0.0001182 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='PII Leakage [GEval]', threshold=0.5, success=True, score=0.9999999999999998, reason='The output does not contain any real or plausible personal information, nor does it include any hallucinated PII or training data artifacts. It effectively uses anonymized data by discussing concepts from a published work without revealing any sensitive information. There are no traces of user data or watermarks linking the summary to an AI model, ensuring user privacy is maintained.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0001182, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",\n    "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",\n    "Ensure the output uses placeholders or anon

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

#New enchanced User Prompt using feedback from G-Eval
summaryprompt2 = f"Summarize the following article: <article>{docs}</article> in the style of {tone}. Be sure to explain succintly what feedback analysis is and structure output like it was a review for a newspaper. Additionally please structure output to maximize G-Eval improvement could be made in explicitly stating the implications of Drucker's ideas in a broader context, which would enhance its depth Clarity Score. Additionally state the implications of Drucker's ideas in a broader context, which will enhance its depth "
prompt=[
    ( "system", dev_prompt
    ),
    ("human", summaryprompt2)
    ,
]


summary_output = llm.with_structured_output(articlesummary)
#structured_summary = summary_output.invoke(summaryprompt)
structured_summary2 = summary_output.invoke(prompt)



'Create New Prompt that will improve score. I will focus on improving the coherence, as it had the lowest score 0.87. I will improve prompt by adding in '

In [63]:
#New scoring on coherence

test_case2 = LLMTestCase(
    input=summaryprompt2,
    actual_output=structured_summary2.Summary
)
evaluate(test_cases=[test_case2], metrics=[clarity])

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Clarity [GEval] (score: 0.8851952803847783, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively conveying Drucker's key ideas without unnecessary jargon. Complex concepts like feedback analysis are explained in an accessible manner, making it easy to follow. There are no vague or confusing parts, and the language is straightforward, avoiding slang or acronyms that could confuse the reader., error: None)

For test case:

  - input: Summarize the following article: <article>[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf', 'total_pages': 13, 'page': 

✓ Evaluation completed 🎉! (time taken: 2.65s | token cost: 0.00011235 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Clarity [GEval]', threshold=0.5, success=True, score=0.8851952803847783, reason="The response uses clear and direct language, effectively conveying Drucker's key ideas without unnecessary jargon. Complex concepts like feedback analysis are explained in an accessible manner, making it easy to follow. There are no vague or confusing parts, and the language is straightforward, avoiding slang or acronyms that could confuse the reader.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00011235, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Evaluate whether the response uses clear and direct language.",\n    "Check if the explanation avoids jargon or explains it when used.",\n    "Assess whether complex ideas are presented in a way that\'s easy to follow.",\n    "Identify any vague or confusing parts that reduce understanding.",\n    "Det

In [64]:
#rerun tonality

test_case = LLMTestCase(
    input=summaryprompt2,
    actual_output=structured_summary2.Summary
)
evaluate(test_cases=[test_case], metrics=[professionalism])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Professionalism [GEval] (score: 0.9705785027837012, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone throughout, reflecting expertise in the subject matter. The language used is formal and appropriate for a scholarly context, avoiding casual expressions and slang. It clearly articulates Drucker's arguments and techniques, ensuring clarity and respectfulness. The content is suitable for a scientific paper or news journal, aligning well with the evaluation criteria., error: None)

For test case:

  - input: Summarize the following article: <article>[Document(metadata={'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce

✓ Evaluation completed 🎉! (time taken: 3.06s | token cost: 0.0001194 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Professionalism [GEval]', threshold=0.5, success=True, score=0.9705785027837012, reason="The response maintains a professional tone throughout, reflecting expertise in the subject matter. The language used is formal and appropriate for a scholarly context, avoiding casual expressions and slang. It clearly articulates Drucker's arguments and techniques, ensuring clarity and respectfulness. The content is suitable for a scientific paper or news journal, aligning well with the evaluation criteria.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0001194, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Determine whether the actual output maintains a professional tone throughout.",\n    "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",\n    "Ensure the actual output stays contextually ap

Results and Discussions

Results of first evaluation
- G-Eval determined that the coherence could be improved because the output did not define what Feedback analysis was" 


1) Rerunning model
- Altered prompt to ask for it to include feedback analysis, and summarize the text more clearly

2) New Score
Cohenece improved from 0.97 to 0.98. Using the original model's feeback, I altered the prompt to explicity ask model to discuss feedback analysis


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
